# Resumizer RAG pipeline
This RAG Pipeline uses the Kaggle Resume dataset. 
The architecture is a simple
 - A vector database
 - A Locally running ollama LLM
 - Embedding model that embeds the chunks
 - RAG pipeline that sends only relevant chunks from vector in context to LLM for evaluation
 - Finally we have some test set to get the Accuracy.

In [1]:
%pip install langchain-community pypdf
%pip install chromadb
%pip install sentence-transformers langchain-huggingface
%pip install ragas pandas 
%pip install rapidfuzz
%pip install langchain-ollama

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached pypdf-6.9.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-1.2.22-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_classic-1.0.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached sqlalchemy-2.0.48-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (9.5 kB)
  Using cached requests-2.33.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached aiohttp-3.13.3-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached langsmith-0.7.22-py3-none-any.whl.m

In [2]:
# Test to see if the local llm is running and reachable.
from langchain_ollama import ChatOllama

local_llm = ChatOllama(
    model="qwen2.5:3b", 
    # Dont' make things up.
    temperature=0,
)

# Make sure the model name matches exactly what you see in `ollama list`
result = local_llm.invoke("Do parrots talk?")

print(result.content)


/home/rvv/Downloads/Resumizer/resumerag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/home/rvv/Downloads/Resumizer/resumerag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Yes, many species of parrots can mimic human speech and other sounds to some extent. They have the ability to learn and repeat various vocalizations, including words or phrases that they hear often enough. However, it's important to note that not all parrots will be able to talk as well as others, and their abilities can vary greatly depending on the species and individual bird. Some parrot species are better known for their speech capabilities than others.


In [3]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
# Path to your main folder
path = "./data/data/data/INFORMATION-TECHNOLOGY"

# glob="**/*.pdf" tells it to look in all subfolders recursively
loader = DirectoryLoader(
    path, 
    glob="**/*.pdf", 
    loader_cls=PyPDFLoader,
    show_progress=True, # Shows a handy progress bar
    use_multithreading=True # Speeds up loading for large volumes
)

docs = loader.load()
print(f"# of documents loaded {len(docs)}")

100%|█████████████████████████████████████████| 120/120 [00:24<00:00,  4.93it/s]

# of documents loaded 247


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

chunks = text_splitter.split_documents(docs)

print(f"# of chunkds {len(chunks)}")

# of chunkds 1092


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os
import shutil

chrom_db_path = "/tmp/chroma_db/"
# 1. FORCE DELETE the old folder to be 100% sure

if os.path.exists(chrom_db_path):
    print("Deleting existing")
    shutil.rmtree(chrom_db_path)
# "all-MiniLM-L6-v2" is fast, small, and very accurate for its size
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
abs_path = os.path.abspath(chrom_db_path)
print(f"Creating new vector db at path {abs_path}")
# Now use it exactly like you did before
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings,
    persist_directory=abs_path
)

if(vectorstore is not None):
    print(f"Vector store initalized")

Deleting existing


Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 6484.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating new vector db at path /tmp/chroma_db
Vector store initalized


In [6]:
# Verify it worked
print(f"Number of chunks: {len(chunks)}")
count = vectorstore._collection.count()
if(len(chunks)==count):
    print("The vector db row count and number of chunks are equal")
else:
    raise ValueError("Corrupt db")

Number of chunks: 1092
The vector db row count and number of chunks are equal


In [7]:
query = "Find all candidates with experience in Computer science"
relevant_chunks = vectorstore.similarity_search(query, k=5)

for i,doc in enumerate(relevant_chunks):
    print(f"\n-------------------- Result {i+1} ---------------------------")
    print(f"Content: {doc.page_content[:400]}...") # Print first 200 chars
    print(f"Metadata: {doc.metadata}")


-------------------- Result 1 ---------------------------
Content: INFORMATION TECHNOLOGY SPECIALIST
Professional Profile
Quality-driven and practical Systems Administrator with [Number] years aligning business systems with business policies and guidelines. Looking
to bring strong analytical and problem-solving skills to an industry-leading software company.
Qualifications
CompTIA Security + CE SY0-401
Certified
Refined system debugging and
diagnostic skills
Exce...
Metadata: {'source': 'data/data/data/INFORMATION-TECHNOLOGY/22450718.pdf', 'creationdate': '2021-08-08T15:32:47+05:30', 'title': '', 'page_label': '1', 'creator': 'wkhtmltopdf 0.12.4', 'producer': 'Qt 4.8.7', 'total_pages': 2, 'page': 0}

-------------------- Result 2 ---------------------------
Content: INFORMATION TECHNOLOGY MANAGER
Qualifications
Strong communication skills Web application design
Working independently HTML
Leadership Adobe Acrobat Professional
IT Governance 
Adobe Photoshop
Requirements gathering 
Adobe

In [8]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Convert existing Chroma vector store into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

system_prompt = (
    "You are an assistant for recruiting tasks. "
    "Use the following pieces of retrieved resumes to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 4. Create the helper chain that "stuffs" the docs into the prompt
question_answer_chain = create_stuff_documents_chain(local_llm, prompt)

# 5. Create the final retrieval chain
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "Which candidates have more than 3 years of experience?"})

print(response["answer"])


Based on the information provided in the resume, Jose Miguel Morales has over 8 years of experience as an Information Technology Manager. However, there is no specific mention of having more than 3 years of experience for any other candidate or skill listed. Therefore, we cannot determine if any other candidates have more than 3 years of experience based solely on this information.


In [9]:
response = rag_chain.invoke({"input": "How many candidate has management experience. Provide the resume name"})
print(response["answer"])

Based on the information provided in the resumes, only one candidate is explicitly mentioned to have management experience:

INFORMATION TECHNOLOGY MANAGER - Responsible for System Implementations, Project Management, and Customer Executive Level communications.

The TRAINING MANAGER resume does not mention any specific management experience.


In [10]:
# This cell is extraneous but demonstrates how we can generate.

from rapidfuzz import process, fuzz

def find_resume_chunks(query, all_chunks, limit=3):
    """
    Search through your resume chunks to find evidence for your test cases.
    """
    # Extract just the text from your LangChain Document objects
    chunk_texts = [doc.page_content for doc in all_chunks]
    
    # Perform a fuzzy search
    results = process.extract(
        query, 
        chunk_texts, 
        scorer=fuzz.partial_ratio, 
        limit=limit
    )
    
    print(f"\n--- Top {limit} Matches for: '{query}' ---")
    for i, (text, score, index) in enumerate(results):
        source = all_chunks[index].metadata.get('source', 'Unknown File')
        print(f"\n[Match {i+1}] (Score: {int(score)}) | Source: {source}")
        print(f"{text[:300]}...") # Print first 300 chars
        print("-" * 30)
    
    return [all_chunks[i] for _, _, i in results]


kotlin_javascript = find_resume_chunks("Kotlin and Javascript", chunks)
experience = find_resume_chunks("20+ years of experience", chunks)
cobol = find_resume_chunks("cobol", chunks)




--- Top 3 Matches for: 'Kotlin and Javascript' ---

[Match 1] (Score: 76) | Source: data/data/data/INFORMATION-TECHNOLOGY/10641230.pdf
Programming and Web 
troubleshooting q 
HTML - HTML5 
q 
Optimizing and performance tuning q 
XML 
q 
Audio and video technologies q
CSS - CSS3 
q 
Medical technology installation and q 
JavaScript 
troubleshooting q 
Command Line q 
Java 
Management q 
ActionScript 
q
Hardware and software upgrade ...
------------------------------

[Match 2] (Score: 71) | Source: data/data/data/INFORMATION-TECHNOLOGY/12635195.pdf
Objective
To obtain a position in the information technology, personnel development, or computer science field to help manage, develop, and support
projects and individuals.
ADJUNCT INFORMATION TECHNOLOGY INSTRUCTOR
Experience
Adjunct Information Technology Instructor
 
01/2014
 
to 
Current
 
Compa...
------------------------------

[Match 3] (Score: 71) | Source: data/data/data/INFORMATION-TECHNOLOGY/12635195.pdf
Taught programming courses

In [11]:
import csv
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import PromptTemplate

if(local_llm is not None):
    print("LLM Initalized")

test_dataset = [
    {
        "question": "Which candidates has over 8 years of experience?",
        "reference_answer": "Jose Miguel Morales has over 8 years of experience"
    },
    {
        "question": "How many candidates have management experience or studied it?",
        "reference_answer": "Three candidate have studied management"
    }
]

eval_prompt = PromptTemplate.from_template(
    "You are an expert evaluator for a search pipeline.\n"
    "Your task is to check if the Generated Answer semantically entails the Reference Answer.\n\n"
    "Rules:\n"
    "1. If the Generated Answer contains the core facts, meaning, or mathematical equivalent (e.g., '10 years' implies 'over 8 years'), it is a match.\n"
    "2. Extra information in the Generated Answer is perfectly fine. Ignore it.\n\n"
    "Example 1:\n"
    "Reference: The sky is blue.\n"
    "Generated: Based on the documents, the sky is blue during the day, but turns orange at sunset.\n"
    "Reasoning: The generated answer contains the core fact that the sky is blue, plus extra context.\n"
    "Verdict: YES\n\n"
    "Example 2:\n"
    "Reference: John has over 5 years of experience.\n"
    "Generated: John has 7 years of experience in the industry.\n"
    "Reasoning: 7 years is mathematically over 5 years, so the core fact is satisfied.\n"
    "Verdict: YES\n\n"
    "Now evaluate the following:\n"
    "Question: {question}\n"
    "Reference Answer: {reference_answer}\n"
    "Generated Answer: {generated_answer}\n\n"
    "First, provide a 1-sentence reasoning. Then, on a new line, provide the final verdict as exactly 'Verdict: YES' or 'Verdict: NO'."
)
# Create the evaluation chain
eval_chain = eval_prompt | local_llm

LLM Initalized


In [ ]:
def resume_rag_pipeline(question):
    if(rag_chain is None):
        raise ValueError("rag_chain is not initalized")
    return rag_chain.invoke({"input": question})

def run_local_evaluation(dataset, output_filename="rag_evaluation_results.csv"):
    results = []
    
    print(f"Starting evaluation of {len(dataset)} questions...\n")
    
    for i, item in enumerate(dataset):
        question = item["question"]
        reference = item["reference_answer"]
        
        # Step A: Get the answer from your RAG pipeline
        print(f"Running Q{i+1}: {question}")
        generated_output = resume_rag_pipeline(question)
        # Extract the actual text string from the LangChain dictionary
        # (It falls back to stringifying the whole thing just in case it's not a dict)
        if isinstance(generated_output, dict) and "answer" in generated_output:
            generated_text = generated_output["answer"]
        else:
            generated_text = str(generated_output)
        print(f"RAG Generated response Q{i+1}: {generated_text}")
        print(f"Test Reference answer Q{i+1}: {reference}")
        
        # Step B: Have Ollama grade the response
        judge_response = eval_chain.invoke({
            "question": question,
            "reference_answer": reference,
            "generated_answer": generated_text
        })
        
        judge_text = judge_response.content.strip()
        print(f"Judge answer Q{i+1}: {judge_text}")
       
        # Step C: Parse the pass/fail score
        # Look for the exact verdict string we requested
        score = 1 if "VERDICT: YES" in judge_text.upper() else 0
        # Step D: Store the row data
        results.append({
            "Question": question,
            "Reference Answer": reference,
            "Generated Answer": generated_text,
            "Score": score,
            "Judge Reasoning": judge_text
        })
        
    # Step E: Write everything to a local CSV
    with open(output_filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["Question", "Reference Answer", "Generated Answer", "Score", "Judge Reasoning"])
        writer.writeheader()
        writer.writerows(results)
        
    print(f"\nEvaluation complete! Results saved to {output_filename}")
    
    # Calculate a quick summary score
    total_score = sum(r["Score"] for r in results)
    print(f"Final Accuracy: {total_score}/{len(dataset)} ({(total_score/len(dataset))*100:.1f}%)")


run_local_evaluation(test_dataset)
print("Done!")

Starting evaluation of 2 questions...

Running Q1: Which candidates has over 8 years of experience?
RAG Generated response Q1: Based on the information provided in the resumes, Jose Miguel Morales has over 10 years of experience. The candidate's resume states "20+ years' experience" for several IT-related skills such as Series 7 and DOS.

For the second candidate, there is no specific mention of their total years of experience. Therefore, I cannot determine if they have over 8 years of experience based on the given information.
Test Reference answer Q1: Jose Miguel Morales has over 8 years of experience
Judge answer Q1: Reasoning: The Generated Answer states that Jose Miguel Morales has over 10 years of experience, which implies he has over 8 years of experience.
Verdict: YES
Running Q2: How many candidates have management experience or studied it?
